# Example: one site from download to AERONET validation

Edit the paths and site in the first cell. Every step resumes if rerun.

In [ ]:
import os
import celimes_air_quality_package as cap
from celimes_air_quality_package import plotting

SITE = "Reno"
LAT, LON = 39.5413, -119.8136
START, END = "2023-08-01", "2023-08-31"

GOES_DIR  = r"Z:\Nasa_Project\GOES_ABI"
SITE_DIR  = rf"Z:\Nasa_Project\GOES_ABI\resampled_pixels\Sites\{SITE}"
HRRR_NC   = rf"Z:\Nasa_Project\HRRR\{SITE}\{SITE}_goes_hrrr_2023.nc"
AERONET   = rf"Z:\Nasa_Project\AERONET_AOD\sites\{SITE}\20200101_20241231_{SITE}.lev20"

FOOTPRINT = os.path.join(SITE_DIR, f"GOES_ABI_footprint_{SITE.lower()}.csv")
REGRIDDED = os.path.join(SITE_DIR, f"{SITE}_AOD_regridded.csv")
FILLED    = os.path.join(SITE_DIR, f"{SITE}_Gap_filled.csv")

## Credentials
Keys are read from environment variables, so none are stored in this notebook. Set them once in your OS, or for this session only:

```python
os.environ["AIRNOW_API_KEY"] = "..."   # don't save the notebook with a real key in it
```
TEMPO uses `~/.netrc`: run `cap.get_earthdata_credentials()` once.

## 1. Download GOES-West AOD

In [ ]:
cap.download_goes_aod(START, END, GOES_DIR)

## 2. Footprint

In [ ]:
fp = cap.build_goes_footprint(os.path.join(GOES_DIR, "2023", "213", "18"),
                              center=(LAT, LON), half_width_km=40, out_csv=FOOTPRINT)
plotting.plot_footprint(fp, target=(LAT, LON))

## 3. Regrid GOES AOD onto the footprint

In [ ]:
cap.regrid_goes_aod(GOES_DIR, FOOTPRINT, REGRIDDED, START, END)

## 4. HRRR at the footprint pixels
Runs in fresh subprocesses of 20 days each. Check a variable string first with `cap.verify_hrrr_variable(FOOTPRINT, "surface", "HPBL")` if you add new fields.

In [ ]:
cap.run_hrrr_supervised(FOOTPRINT, HRRR_NC, "2023-01-01", "2023-12-31")

## 5. Merge and gap fill

In [ ]:
merged, hrrr_df = cap.combine_goes_hrrr(REGRIDDED, HRRR_NC, start=START, end=END)
cap.goes_aod_gap_filling(merged, FILLED, START, END,
                         target_point=(LAT, LON), save_radius_km=30)

In [ ]:
import pandas as pd
filled = pd.read_csv(FILLED)
plotting.plot_hour(filled, "filled_AOD", "2023-08-15 20:00")

## 6. Validate against AERONET

In [ ]:
ae = cap.load_aeronet_lev20(AERONET)
raw = cap.extract_raw_goes_at_site(GOES_DIR, LAT, LON, START, END)
filled_site = cap.extract_filled_at_site(FILLED, LAT, LON, radius_km=7.5, use="filled")

res = cap.validate_site(raw, filled_site, ae, threshold=25)
print(res["stats_raw"])
print(res["stats_filled"])
plotting.plot_validation_scatter(res["raw_merged"], res["filled_merged"],
                                 titles=[f"(a) Native Retrievals: {SITE}", f"(b) Gap-Filled Retrievals: {SITE}"])

In [ ]:
cap.threshold_sensitivity(raw, filled_site, ae, thresholds=(25, 50, 75), site=SITE)

## TEMPO NO2 (same pattern)

In [ ]:
TEMPO_DIR = r"Z:\Nasa_Project\TEMPO_NO2\NO2_retrievals\L3_data\organized"
# cap.download_tempo("2024-01-01", "2024-01-31", TEMPO_DIR)
# tfp = cap.build_tempo_footprint(os.path.join(TEMPO_DIR, "2024", "01", "03", "19"), center=(LAT, LON), half_width_km=30)
# cap.regrid_tempo(TEMPO_DIR, tfp, os.path.join(SITE_DIR, f"Resampled_TEMPO_NO2_{SITE}.h5"))
# merged_no2, hrrr_df = cap.combine_tempo_hrrr(os.path.join(SITE_DIR, f"Resampled_TEMPO_NO2_{SITE}.h5"), HRRR_NC)
# cap.tempo_no2_gap_filling(merged_no2, os.path.join(SITE_DIR, f"{SITE}_filled_no2.csv"),
#                           "2023-08-01", "2024-12-31", hrrr_data=hrrr_df)